# Semantic MOCs: From Textual Content to GenAI Applications

This is the **second tutorial** in the TextualMOC series. While [Tutorial 1](https://github.com/ggreco77/TextualMOC/tree/main/tuto1_TextualMOC) introduced the basics of creating and handling Textual MOCs, this notebook explores the **semantic** dimension — transforming textual content into vector embeddings for Generative AI (GenAI) applications.

The notebook is associated with the paper **[Encapsulating Textual Contents into a MOC data structure for Advanced Applications](https://www.sciencedirect.com/science/article/pii/S2213133725000873)**; Greco et al., 2026.

### Ollama models

```bash
ollama pull nomic-embed-text && ollama pull mxbai-embed-large && ollama pull snowflake-arctic-embed
ollama pull mistral && ollama pull gemma3:4b
```


## What is RAG?

**Retrieval-Augmented Generation (RAG)** first **retrieves** relevant documents via semantic search, then passes them as context to an LLM which **generates** an answer grounded in real data — reducing hallucinations.

In our case, each "document" is a Textual MOC carrying both text *and* a sky region, enabling RAG that is simultaneously **semantic** and **spatial**.

## Setup

In [1]:
#conda create -n acme_tuto_vo_genai python=3.12 -y
#conda activate acme_tuto_vo_genai

#python -m pip install --upgrade pip setuptools wheel

#python -m pip install \
#  mocpy==0.20.0 \
#  matplotlib==3.10.8 \
#  numpy==2.4.3 \
#  requests==2.32.5 \
#  beautifulsoup4 \
#  ipyaladin==0.3.0 \
#  ipywidgets==8.1.8 \
#  astropy==7.2.0 \
#  pandas==3.0.1 \
#  langchain-community==0.4.1 \
#  langchain-ollama==1.0.1 \
#  scikit-learn \
#  astroquery==0.4.12.dev10525 \
#  cdshealpix==0.8.1 \
#  notebook==7.5.5 \
#  ipykernel==7.2.0 \
#  transformers==5.3.0

#python -m ipykernel install --user \
#  --name acme_tuto_vo_genai \
#  --display-name "Python (acme_tuto_vo_genai)"

In [2]:
import mocpy, matplotlib, numpy, requests, bs4, ipywidgets, astropy, pandas, ipyaladin, astroquery, \
       langchain_community,langchain_ollama,ipykernel, transformers
for mod in [mocpy, matplotlib, numpy, requests, bs4, ipywidgets, astropy, pandas,ipyaladin, astroquery,
            langchain_community, langchain_ollama,ipykernel, transformers]:
    print(f"{mod.__name__:20s} {mod.__version__}")


PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


mocpy                0.20.0
matplotlib           3.10.8
numpy                2.4.3
requests             2.32.5
bs4                  4.14.3
ipywidgets           8.1.8
astropy              7.2.0
pandas               3.0.1
ipyaladin            0.3.0
astroquery           0.4.12.dev10525
langchain_community  0.4.1
langchain_ollama     1.0.1
ipykernel            7.2.0
transformers         5.3.0


In [3]:
import sys
print(sys.version)

3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:51:19) [Clang 19.1.7 ]


## Import the `enriched_moc` module

In [4]:
import requests
from pathlib import Path

REQUIRED_VERSION = "0.0.7"
url = "https://raw.githubusercontent.com/ggreco77/TextualMOC/main/textualmoc/enriched_moc.py"
dest = Path("enriched_moc.py")

def get_local_version():
    if not dest.exists(): return None
    for line in dest.read_text().split('\n'):
        if '__version__' in line: return line.split('"')[1]
    return "unknown"

local = get_local_version()
if local is None:
    r = requests.get(url, timeout=30); r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"Downloaded enriched_moc.py v{REQUIRED_VERSION}")
elif local == REQUIRED_VERSION:
    print(f"enriched_moc.py v{local} already up to date.")
else:
    print(f"Local version: {local} — required: {REQUIRED_VERSION}")

from enriched_moc import TextualMOC, SemanticMOC

enriched_moc.py v0.0.7 already up to date.


## Overview of SemanticMOC methods

In [5]:
import inspect
from IPython.display import display, Markdown
import pandas as pd

def show_methods(cls):
    methods = []
    for name, method in inspect.getmembers(cls, predicate=inspect.isfunction):
        if name.startswith('_'): continue
        doc = (method.__doc__ or '').strip().split('\n')[0]
        inherited = name not in cls.__dict__
        source = f" *(from {cls.__bases__[0].__name__})*" if inherited else ''
        methods.append({'Method': name, 'Description': doc + source})
    df = pd.DataFrame(methods).sort_values(by='Method').reset_index(drop=True)
    df.index = df.index + 1; df.index.name = 'No.'
    pd.set_option('display.max_colwidth', None)
    display(Markdown(f'# Methods in `{cls.__name__}`')); display(df)

show_methods(SemanticMOC)

# Methods in `SemanticMOC`

,Method,Description
No.,,
1,add_text_media_image,"Adds custom text, a multimedia link, and an image link to the MOC's *(from TextualMOC)*"
2,annotate_cell,Assigns a textual annotation to a specific MOC cell. *(from TextualMOC)*
3,embedding_from_custom_text,Generates an embedding from the 'text' key in moc_data using
4,load_semantic_moc,Loads a Semantic MOC from a JSON file.
5,load_textual_moc,Loads a Textual MOC from a JSON file. *(from TextualMOC)*
6,plot_embedding,Plots the embedding vector as a bar chart.
7,plot_moc_area,Visualizes the MOC area using matplotlib. Annotated cells are *(from TextualMOC)*
8,render,Loads the MOC from a JSON file and displays its contents. *(from TextualMOC)*
9,render_ipyaladin,Visualize the MOC in an Aladin viewer. Annotated cells are *(from TextualMOC)*


## From Textual MOC to Semantic MOC

A **Semantic MOC** = Textual MOC + embedding vector.

> **Ensure:** Ollama is running: `ollama pull nomic-embed-text`

In [6]:
semantic_moc = SemanticMOC()
semantic_moc.load_semantic_moc("textual_moc_example.json")
semantic_moc.embedding_from_custom_text(embeddings_model="nomic-embed-text")
semantic_moc.show_embedding()
semantic_moc.save("moc_data_with_embedding.json")

Textual MOC loaded from textual_moc_example.json.
  └─ No embedding found in this file.


/Users/gius/Desktop/ACME/enriched_moc.py:465: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model=embeddings_model)


Embedding added to moc_data using the model: nomic-embed-text
Embedding model: nomic-embed-text
Embedding dimension: 768
Embedding (first 5 values): [0.8172144293785095, 0.558133065700531, -3.155940055847168, -1.6662379503250122, 0.7443456649780273]
Data successfully saved to moc_data_with_embedding.json.


## Managing a Semantic MOC

In [7]:
semantic_moc_loaded = SemanticMOC()
semantic_moc_loaded.load_semantic_moc("moc_data_with_embedding.json")
semantic_moc_loaded.show_text_value()
semantic_moc_loaded.show_metadata_value()
semantic_moc_loaded.show_embedding()

Textual MOC loaded from moc_data_with_embedding.json.
  └─ Embedding found: model=nomic-embed-text, dimensions=768
Custom Text: This is a sample text added to a spatial MOC in JSON format. The text demonstrates how additional information can be embedded into a Multi-Order Coverage (MOC) object, enhancing its descriptive capabilities by including custom textual content alongside the spatial data.
This approach is described in the paper 'Encapsulating Textual Content into MOC data structure for Advanced Applications'.
Author: GG
Date: 2026-03-23
Last Text Update: 2026-03-23 22:05:50
Embedding model: nomic-embed-text
Embedding dimension: 768
Embedding (first 5 values): [0.8172144293785095, 0.558133065700531, -3.155940055847168, -1.6662379503250122, 0.7443456649780273]


## Data Generation for GenAI Applications

We build a collection of Textual MOCs for 11 well-known galaxies.

### Create spatial MOCs from SIMBAD coordinates

In [8]:
import os
import json
import matplotlib.pyplot as plt
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord, Longitude, Latitude, Angle
import astropy.units as u
from mocpy import MOC

# Object galaxy list
galaxies = ["Arp273", "M59", "NGC4676", "M101", "M60", "NGC4993", 
            "M104", "M82", "NGC4038", "M51", "M87"]

# Building Space MOC at the galaxy position
for galaxy in galaxies:
    result = Simbad.query_object(galaxy)

    if result is not None:
        # Extracting RA/DEC
        ra = result["ra"][0]  
        dec = result["dec"][0]  
        coords = SkyCoord(ra, dec, unit=(u.deg, u.deg))

        print(f"Coordinates "+ galaxy)
        print(f"RA  (Right Ascension) : {coords.ra.deg}°")
        print(f"DEC (Declination)      : {coords.dec.deg}°")

        # --- Step 2: Creating circle MOC with radius = 0.1° ---
        radius = Angle(0.1, u.deg)  # fixed radius
        moc = MOC.from_cone(
            lon=Longitude(coords.ra),
            lat=Latitude(coords.dec),
            radius=radius,
            max_depth=14  # MOC resolution
        )

        # --- Step 3: Creating moc_gals dir ---
        save_dir = "moc_gals"
        os.makedirs(save_dir, exist_ok=True)

        # --- Step 4: Saving MOC in json format ---
        moc_json = moc.serialize(format='json')
        save_path = os.path.join(save_dir, galaxy+".json")

        with open(save_path, "w") as json_file:
            json.dump(moc_json, json_file, indent=4)

        print(f"Space MOC saved in  {save_path}")

    else:
        print("No object in Simbad.")

Coordinates Arp273
RA  (Right Ascension) : 35.36960977733°
DEC (Declination)      : 39.37558926756°
Space MOC saved in  moc_gals/Arp273.json
Coordinates M59
RA  (Right Ascension) : 190.50940890631998°
DEC (Declination)      : 11.64691930771°
Space MOC saved in  moc_gals/M59.json
Coordinates NGC4676
RA  (Right Ascension) : 191.54241666666667°
DEC (Declination)      : 30.731583333333333°
Space MOC saved in  moc_gals/NGC4676.json
Coordinates M101
RA  (Right Ascension) : 210.80242916666668°
DEC (Declination)      : 54.34875°
Space MOC saved in  moc_gals/M101.json
Coordinates M60
RA  (Right Ascension) : 190.91654510497253°
DEC (Declination)      : 11.552691159463055°
Space MOC saved in  moc_gals/M60.json
Coordinates NGC4993
RA  (Right Ascension) : 197.448711984°
DEC (Declination)      : -23.38397612351°
Space MOC saved in  moc_gals/NGC4993.json
Coordinates M104
RA  (Right Ascension) : 189.99763274591663°
DEC (Declination)      : -11.623054494444448°
Space MOC saved in  moc_gals/M104.json
Co

### Define galaxy descriptions, images, and links

In [9]:
##### Getting image from hips2fits - https://alasky.cds.unistra.fr/hips-image-services/hips2fits
Arp273_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.1&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=Arp%20273&format=jpg"
NGC4676_ima= "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.1&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=NGC%204676&format=jpg"
NGC4038_ima ="https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FHST%2FEPO&width=1000&height=1000&fov=0.1&projection=SIN&coordsys=icrs&rotation_angle=0.0&ra=180.4790760656&dec=-18.884864677&format=jpg"
M51_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.3&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M51&format=jpg"
M101_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.35&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M101&format=jpg"
M104_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.2&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M104&format=jpg"
M87_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.15&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M87&format=jpg"
M59_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.1&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M59&format=jpg"
M60_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.15&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M60&format=jpg"
NGC4993_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.04&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=NGC%204993&format=jpg"
M82_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.2&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=M82&format=jpg"
#NGC4449_ima = "https://alasky.cds.unistra.fr/hips-image-services/hips2fits?hips=CDS%2FP%2FDSS2%2Fcolor&width=1200&height=900&fov=0.15&projection=SIN&coordsys=icrs&rotation_angle=0.0&object=NGC%204449&format=jpg"

# Short description text
Arp273_text = "The galaxies' twisted and distorted appearance is due to mutual gravitational tides as the pair engage in \
close encounters. Cataloged as Arp 273 (also as UGC 1810), these galaxies do look peculiar, but interacting galaxies are now \
understood to be common in the universe. Closer to home, the large spiral Andromeda Galaxy is known to be some 2 million light-years away \
and inexorably approaching the Milky Way. In fact the far away peculiar galaxies of Arp 273 may offer an analog of the far future \
encounter of Andromeda and Milky Way. Repeated galaxy encounters on a cosmic timescale ultimately result in a merger into a \
single galaxy of stars. From our perspective, the bright cores of the Arp 273 galaxies are separated by only a little over 100,000 light-years. "

NGC4676_text= "This colliding pair of spiral galaxies is known as 'The Mice' because of the long tails of stars and gas emanating \
from each galaxy. Otherwise known as NGC 4676, they will eventually merge into a single giant galaxy. In the galaxy at left, \
the bright blue patch can be identified as a cascade of clusters and associations of young, hot blue stars, whose \
formation has been triggered by the tidal forces of the gravitational interaction. Streams of material can also be seen \
flowing between the two galaxies in this Hubble Space Telescope image. \
The clumps of young stars in the long, straight tidal tail (upper right) are separated by fainter regions \
of material. These dim regions suggest that the clumps of stars have formed from the gravitational \
collapse of the gas and dust that once occupied those areas. Some of the clumps have luminous masses \
comparable to dwarf galaxies that orbit in the halo of our own Milky Way."

NGC4038_text = "This new NASA Hubble Space Telescope image of the Antennae galaxies is the sharpest yet \
of this merging pair of galaxies. During the course of the collision, billions of stars will be formed. The brightest \
and most compact of these star birth regions are called super star clusters. The two spiral galaxies started to interact \
a few hundred million years ago, making the Antennae galaxies one of the nearest and youngest examples of a pair of colliding galaxies. \
Nearly half of the faint objects in the Antennae image are young clusters containing tens of thousands of stars. \
The orange blobs to the left and right of image center are the two cores of the original galaxies and consist mainly \
of old stars criss-crossed by filaments of dust, which appears brown in the image. The two galaxies are dotted with brilliant \
blue star-forming regions surrounded by glowing hydrogen gas, appearing in the image in pink."

M51_text = "M 51, an interacting spiral galaxy, which is also known as the Whirlpool Galaxy. It is located about 25 Million light \
years away from Earth, but can still easily observed with a small telescope by amateur astronomers. M 51 is also a popular object among \
professional astronomers as it shows an ongoing enhanced star formation rate, which is probably caused by the interaction with its \
companion galaxy. The galaxy was also the location of two supernovae within the last couple of years: \
The first one appeared in 2005, the second one in 2011."

M101_text = "Messier 101 is a classic, face-on, pinwheel spiral galaxy. The giant spiral disk of stars, dust and gas is 170,000 light-years across — nearly \
twice the diameter of our galaxy, the Milky Way. M101 is estimated to contain at least one trillion stars. The galaxy’s spiral arms are \
sprinkled with large regions of star-forming nebulas. These nebulas are areas of intense star formation within giant molecular \
hydrogen clouds. Brilliant, young clusters of hot, blue, newborn stars trace out the spiral arms. Pierre Méchain, one of \
Charles Messier’s colleagues, discovered the Pinwheel galaxy in 1781. Located 25 million light-years away from Earth in \
the constellation Ursa Major, M101 has an apparent magnitude of 7.9. It can be spotted through a small telescope and is \
most easily observed during June."

M104_text = "One of most famous spiral galaxies is Messier 104, widely known as the 'Sombrero' (the Mexican hat) because of \
its particular shape. It is located towards the constellation Virgo, at a distance of about 30 million light-years \
and is the 104th object in the famous catalogue of deep-sky objects by French astronomer Charles Messier (1730 - 1817).\
This luminous and massive galaxy has a total mass of about 800 billion suns, and is notable for its dominant nuclear bulge,\
composed mainly of mature stars, and its nearly edge-on disc composed of stars, gas, and dust. The complexity of this dust \
is apparent directly in front of the bright nucleus, but is also evident in the dark absorbing lanes throughout the disc. \
A large number of small, diffuse objects can be seen as a swarm in the halo of Messier 104. Most of these are globular clusters,\
similar to those found in our own Milky Way, but Messier 104 has a much larger number of them."

M87_text = "The elliptical galaxy M87 is the home of several trillion stars, a supermassive black hole and a family of roughly\
15,000 globular star clusters. For comparison, our Milky Way galaxy contains only a few hundred billion stars and about 150 globular \
clusters. The monstrous M87 is the dominant member of the neighboring Virgo cluster of galaxies, which contains some 2,000 galaxies. \
Discovered in 1781 by Charles Messier, this galaxy is located 54 million light-years away from Earth in the constellation Virgo. \
It has an apparent magnitude of 9.6 and can be observed using a small telescope most easily in May."

M59_text = "M59 is one of the largest elliptical galaxies in the Virgo galaxy cluster. However, it is still considerably less massive, \
and at a magnitude of 9.8, less luminous than other elliptical galaxies in the cluster. A supermassive black hole around 270 million times \
as massive as the Sun resides at the center of M59. The galaxy also has an inner disk of stars and around 2,200 globular clusters, \
an exceptionally high number of such clusters. The central region of the galaxy, the inner 200 light-years, rotates in the opposite \
direction than the rest of the galaxy and is the smallest region in a galaxy known to exhibit this behavior. \
Approximately 60 million light-years from Earth, M59 can be found near M58 and M60 in the constellation Virgo. It is best seen in May. \
Small telescopes might reveal an ellipsoidal shape with a bright center, but even larger scopes do not reveal much detail. \
German astronomer Johann Gottfried Koehler discovered M59 and the nearby galaxy M60 in the spring of 1779 when observing the comet \
of that year (Comet Bode)."

M60_text = "The Virgo cluster is a collection of more than 1,300 galaxies, including the elliptical galaxy M60. Unlike spiral galaxies, \
elliptical galaxies lack an organized structure and are nearly featureless, resembling the core of a spiral galaxy. \
The Virgo cluster’s third brightest member, M60 has a diameter of 120,000 light-years and is as massive as one trillion suns. \
At its center lies a huge black hole, 4.5 billion times as massive as the sun — one of the most massive black holes ever found. \
NGC 4647 is about two-thirds the size of M60 — or roughly the size of the Milky Way galaxy — and is much less massive. \
The two galaxies form a pair known as Arp 116. Astronomers have long tried to determine whether these two galaxies are actually interacting. \
Although from Earth they appear to overlap,there is no evidence of new star formation, which would be one of the clearest signs that \
the two galaxies are indeed interacting. However, recent studies of very detailed Hubble images suggest the onset of some tidal \
interaction between the two."

NGC4993_text = "The elliptical galaxy NGC 4993, about 130 million light-years from Earth, viewed with the VIMOS instrument on the European Southern Observatory's Very Large Telescope \
in Chile.After the almost simultaneous detection of gravitational waves by the LIGO/Virgo collaboration, GW170817, and of a gamma-ray burst \
by ESA's INTEGRAL and NASA's Fermi satellites, GRB170817, a large number of ground and space telescopes started searching for the source in the sky. \
About half a day later, scientists at various optical observatories spotted something new near the core of galaxy NGC 4993: \
this was the visible light counterpart to the gravitational waves and the gamma-ray burst, confirming that they originated \
from the collision of two neutron stars. The result of such a cosmic clash is a kilonova: the neutron-rich material released \
in the merger is impacting its surroundings, forging a wealth of heavy elements in the process. The kilonova can be seen just above \
and slightly to the left of the centre of the galaxy, AT2017gfo."

M82_text = "Located 12 million light-years away, M82 appears high in the northern spring sky in the direction of the constellation \
Ursa Major, the Great Bear. It is also called the 'Cigar Galaxy' because of the elongated elliptical shape produced by \
the tilt of its starry disk relative to our line of sight. As shown in this mosaic image, M82 is a magnificent starburst galaxy. \
Throughout its central region young stars are being born ten times faster than they are inside in our Milky Way Galaxy.\
These numerous hot new stars not only emit radiation but also charged particles that form the so-called stellar wind. \
Stellar winds streaming from these stars combine to form a galactic 'superwind'."

#NGC4449_text = ""

# Multimedia URL from text is provided - part of the text has been adapted.
Arp273_media = "https://apod.nasa.gov/apod/ap250109.html"
NGC4676_media= "https://science.nasa.gov/image-detail/idl-tiff-file-40/"
NGC4038_media ="https://hubblesite.org/contents/media/images/2006/46/1995-Image"
M51_media = "https://esahubble.org/images/opo0521b/"
M101_media = "https://science.nasa.gov/mission/hubble/science/explore-the-night-sky/hubble-messier-catalog/messier-101/"
M104_media = "https://www.eso.org/public/images/sombrero/"
M87_media = "https://science.nasa.gov/mission/hubble/science/explore-the-night-sky/hubble-messier-catalog/messier-87/#:~:text=The%20elliptical%20galaxy%20M87%20is,and%20about%20150%20globular%20clusters."
M59_media = "https://science.nasa.gov/mission/hubble/science/explore-the-night-sky/hubble-messier-catalog/messier-59/"
M60_media = "https://science.nasa.gov/mission/hubble/science/explore-the-night-sky/hubble-messier-catalog/messier-60/"
NGC4993_media = "https://sci.esa.int/web/integral/-/59671-new-source-in-galaxy-ngc-4993"
M82_media = "https://www.esa.int/Science_Exploration/Space_Science/Hubble_s_view_of_Cigar_Galaxy_on_sixteenth_mission_anniversary"
#NGC4449_media = ""

# Text description list
text_files = [Arp273_text, M59_text, NGC4676_text, M101_text, M60_text, NGC4993_text, 
              M104_text,M82_text, NGC4038_text, M51_text, M87_text]

# Multimedia link list
multimedia_urls = [Arp273_media, M59_media, NGC4676_media, M101_media, M60_media, NGC4993_media, 
                   M104_media,M82_media, NGC4038_media, M51_media, M87_media]

# Images fron hips2fits list
images = [Arp273_ima, M59_ima, NGC4676_ima, M101_ima, M60_ima, NGC4993_ima, 
          M104_ima,M82_ima,	NGC4038_ima, M51_ima, M87_ima]

# Space MOCs list 
text_moc_gals = ["Arp273.json", "M59.json", "NGC4676.json", "M101.json", "M60.json", "NGC4993.json", 
                 "M104.json","M82.json", "NGC4038.json", "M51.json", "M87.json"]

### Enrich the spatial MOCs

In [10]:
textual_moc = TextualMOC()
import os
from pathlib import Path
mocs_directory = Path("moc_gals")
for text_moc_gal, text_file, multimedia_url, image in zip(
    text_moc_gals, text_files, multimedia_urls, images):
    file_path = os.path.join(mocs_directory, text_moc_gal)
    textual_moc.load_textual_moc(file_path)
    textual_moc.add_text_media_image(text_file, multimedia_url, image)
    textual_moc.save(file_path)
print(f"Enriched {len(text_moc_gals)} Textual MOCs in {mocs_directory}/")

Textual MOC loaded from moc_gals/Arp273.json.
Data successfully saved to moc_gals/Arp273.json.
Textual MOC loaded from moc_gals/M59.json.
Data successfully saved to moc_gals/M59.json.
Textual MOC loaded from moc_gals/NGC4676.json.
Data successfully saved to moc_gals/NGC4676.json.
Textual MOC loaded from moc_gals/M101.json.
Data successfully saved to moc_gals/M101.json.
Textual MOC loaded from moc_gals/M60.json.
Data successfully saved to moc_gals/M60.json.
Textual MOC loaded from moc_gals/NGC4993.json.
Data successfully saved to moc_gals/NGC4993.json.
Textual MOC loaded from moc_gals/M104.json.
Data successfully saved to moc_gals/M104.json.
Textual MOC loaded from moc_gals/M82.json.
Data successfully saved to moc_gals/M82.json.
Textual MOC loaded from moc_gals/NGC4038.json.
Data successfully saved to moc_gals/NGC4038.json.
Textual MOC loaded from moc_gals/M51.json.
Data successfully saved to moc_gals/M51.json.
Textual MOC loaded from moc_gals/M87.json.
Data successfully saved to moc_ga

## RAG Pipeline: From Query to Answer

1. **Index** — load MOCs, chunk texts, compute embeddings
2. **Retrieve** — find relevant chunks by semantic similarity
3. **Generate** — pass **full documents** to the LLM (chunks are only for retrieval)
4. **Visualize** — show the MOCs cited by the LLM in ipyaladin
5. **Analyze** — classify galaxy images with a vision model
6. **Spatial query** — find MOCs by sky position

### Step 1: Building the Vector Index from Text Chunks

Each text is split into **chunks** for retrieval. The chunk finds the relevant MOC; the **full document** is passed to the LLM.

In [11]:
import os
import numpy as np

#from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

from enriched_moc import TextualMOC

# Settings
MOC_DIR = "moc_gals"                  # directory textual MOCs
EMBEDDING_MODEL = "mxbai-embed-large" # embedding model used in Ollama
TOKENIZER_MODEL = "mixedbread-ai/mxbai-embed-large-v1"  # tokenizer used for chunking
SIMILARITY_THRESHOLD = 0.6            # get chunks with similarity > threshold

# Chunking settings
# Here chunk size and overlap are in TOKENS, not words/characters
CHUNK_SIZE = 150
CHUNK_OVERLAP = 20

# Load tokenizer
print(f"Loading tokenizer: {TOKENIZER_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL)

def token_count(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

# Load MOCs
moc_files = sorted([
    os.path.join(MOC_DIR, f)
    for f in os.listdir(MOC_DIR)
    if f.endswith(".json")
])

mocs, filenames, texts = [], [], []

for fp in moc_files:
    m = TextualMOC()
    m.load_textual_moc(fp)

    txt = m.moc_data.get("text", "")
    if txt and getattr(m, "moc", None) is not None:
        mocs.append(m)
        filenames.append(os.path.basename(fp))
        texts.append(txt)

print(f"Loaded {len(mocs)} MOCs")

# Split text into chunks using RecursiveCharacterTextSplitter
# but measuring chunk size in TOKENS
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=token_count,
    is_separator_regex=False,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunk_mocs = []
chunk_filenames = []
chunk_texts = []
chunk_metadata = []

for moc, fn, txt in zip(mocs, filenames, texts):
    docs = text_splitter.create_documents(
        texts=[txt],
        metadatas=[{"filename": fn}]
    )

    for chunk_idx, doc in enumerate(docs):
        content = doc.page_content.strip()
        if not content:
            continue

        n_tokens = token_count(content)

        chunk_mocs.append(moc)
        chunk_filenames.append(fn)
        chunk_texts.append(content)
        chunk_metadata.append({
            "filename": fn,
            "chunk_index": chunk_idx,
            "n_tokens": n_tokens,
            "n_chars": len(content)
        })

print(
    f"Chunked into {len(chunk_texts)} segments "
    f"(chunk_size={CHUNK_SIZE} tokens, overlap={CHUNK_OVERLAP} tokens)"
)

# Optional: inspect chunk stats
if chunk_metadata:
    token_sizes = [meta["n_tokens"] for meta in chunk_metadata]
    print(f"Min chunk tokens: {min(token_sizes)}")
    print(f"Max chunk tokens: {max(token_sizes)}")
    print(f"Average chunk tokens: {sum(token_sizes)/len(token_sizes):.1f}")

# Compute embeddings
print(f"Computing embeddings with {EMBEDDING_MODEL} ...")
ollama_embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

emb_list = []
for i, t in enumerate(chunk_texts):
    emb = ollama_embeddings.embed_query(t)
    emb_list.append(emb)

    if (i + 1) % 10 == 0 or i == len(chunk_texts) - 1:
        print(f"  {i+1}/{len(chunk_texts)}", end="\r")

if len(emb_list) == 0:
    raise ValueError("No embeddings were generated: chunk_texts is empty.")

embeddings = np.array(emb_list, dtype=np.float32)

# Safe normalization
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms[norms == 0] = 1e-12
embeddings = embeddings / norms

print(f"\nEmbeddings: {embeddings.shape}")

# Optional: preview first chunks
print("\nFirst 3 chunks preview:")
for i in range(min(3, len(chunk_texts))):
    print(
        f"\n--- Chunk {i} "
        f"({chunk_metadata[i]['filename']}, {chunk_metadata[i]['n_tokens']} tokens) ---"
    )
    print(chunk_texts[i][:300] + ("..." if len(chunk_texts[i]) > 300 else ""))

Loading tokenizer: mixedbread-ai/mxbai-embed-large-v1 ...
Textual MOC loaded from moc_gals/Arp273.json.
Textual MOC loaded from moc_gals/M101.json.
Textual MOC loaded from moc_gals/M104.json.
Textual MOC loaded from moc_gals/M51.json.
Textual MOC loaded from moc_gals/M59.json.
Textual MOC loaded from moc_gals/M60.json.
Textual MOC loaded from moc_gals/M82.json.
Textual MOC loaded from moc_gals/M87.json.
Textual MOC loaded from moc_gals/NGC4038.json.
Textual MOC loaded from moc_gals/NGC4676.json.
Textual MOC loaded from moc_gals/NGC4993.json.
Loaded 11 MOCs
Chunked into 19 segments (chunk_size=150 tokens, overlap=20 tokens)
Min chunk tokens: 28
Max chunk tokens: 150
Average chunk tokens: 104.8
Computing embeddings with mxbai-embed-large ...
  19/19
Embeddings: (19, 1024)

First 3 chunks preview:

--- Chunk 0 (Arp273.json, 134 tokens) ---
The galaxies' twisted and distorted appearance is due to mutual gravitational tides as the pair engage in close encounters. Cataloged as Arp 273 (also 

### Step 2: Semantic search

For each retrieved result, return the matched chunk (the passage responsible for retrieval) and the full document (the content provided to the LLM).

### Try your own prompt

In [12]:
# Enter your query
query = "Identify the host galaxy of the gravitational-wave event GW170817 in the provided sample and make \
         a short summary about properties and identification of GRB"

#query = "identify the Sombrero  galaxy and provide a short summary "

#query = "find the Mice galaxy in the provided sample and make a short summary"

In [13]:
# for comparison: similarities from klearn.metrics.pairwise
#from sklearn.metrics.pairwise import cosine_similarity  # pip install scikit-learn
#q_vec = np.array(ollama_embeddings.embed_query(query), dtype='float32').reshape(1, -1)
#similarities = cosine_similarity(embeddings, q_vec).flatten()

q_vec = np.array(ollama_embeddings.embed_query(query), dtype='float32')
q_vec = q_vec / np.linalg.norm(q_vec)
similarities = embeddings @ q_vec

ranked = sorted(enumerate(similarities), key=lambda x: -x[1])

print(f"Query: '{query}'")
print('=' * 90 + '\n')

seen, results = set(), []
for idx, sim in ranked:
    fn = chunk_filenames[idx]
    if fn in seen: continue
    seen.add(fn)
    if sim < SIMILARITY_THRESHOLD: continue
    moc = chunk_mocs[idx]
    results.append((moc, float(sim), fn))
    title = os.path.splitext(fn)[0]
    matched = chunk_texts[idx]
    full = moc.moc_data.get('text', '')
    print(f"  Rank {len(results)}: {title} ({fn})  —  sim: {sim:.4f}")
    print(f"  ┌─ MATCHED CHUNK ({len(matched.split())} words): \"{matched[:120]}...\"")
    print(f"  └─ FULL DOCUMENT ({len(full.split())} words): \"{full[:150]}...\"")
    print()
    if len(results) >= 3: break
print(f"{len(results)} result(s)")

Query: 'Identify the host galaxy of the gravitational-wave event GW170817 in the provided sample and make          a short summary about properties and identification of GRB'

  Rank 1: NGC4993 (NGC4993.json)  —  sim: 0.7507
  ┌─ MATCHED CHUNK (67 words): "The elliptical galaxy NGC 4993, about 130 million light-years from Earth, viewed with the VIMOS instrument on the Europe..."
  └─ FULL DOCUMENT (161 words): "The elliptical galaxy NGC 4993, about 130 million light-years from Earth, viewed with the VIMOS instrument on the European Southern Observatory's Very..."

  Rank 2: M87 (M87.json)  —  sim: 0.6747
  ┌─ MATCHED CHUNK (99 words): "The elliptical galaxy M87 is the home of several trillion stars, a supermassive black hole and a family of roughly15,000..."
  └─ FULL DOCUMENT (99 words): "The elliptical galaxy M87 is the home of several trillion stars, a supermassive black hole and a family of roughly15,000 globular star clusters. For c..."

  Rank 3: Arp273 (Arp273.json)  —  sim: 0.6

### Step 3: Generate an answer with RAG

The **full documents** (not chunks) are passed to the LLM as context. Below you can see exactly what the LLM receives: system prompt, user prompt with context, and the response.

In [14]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import os

# Build context from FULL parent documents (not chunks)
contexts = []
used_doc_ids = []

for i, (moc, sim, fn) in enumerate(results, 1):
    full_text = moc.moc_data.get("text", "").strip()
    if not full_text:
        continue

    title = os.path.splitext(fn)[0]
    used_doc_ids.append(str(i))
    contexts.append(
        f"[Doc {i}] (title={title}, similarity={sim:.3f})\n{full_text}"
    )
    print(f"  [Doc {i}] {title} — {len(full_text.split())} words (full document)")

docs_block = "\n\n".join(contexts)
print(f"\nTotal context: {len(docs_block.split())} words from {len(contexts)} documents")

SYSTEM_PROMPT = (
    "You are an astrophysics assistant. "
    "Answer using ONLY the provided documents. "
    "Cite sources inline with [Doc n]. "
    "If the documents do not contain enough information, reply exactly: I don't know."
)

USER_PROMPT = (
    f"Question:\n{query}\n\n"
    f"Documents:\n{docs_block}\n\n"
    f"Provide a concise answer with citations [Doc n]. "
    f"End with a line formatted exactly as: USED: {','.join(used_doc_ids)}"
)

print("\n" + "=" * 60)
print("SYSTEM PROMPT")
print("=" * 60)
print(SYSTEM_PROMPT)

print("\n" + "=" * 60)
print(f"USER PROMPT ({len(USER_PROMPT.split())} words, showing first 600 chars)")
print("=" * 60)
print(USER_PROMPT[:600])
if len(USER_PROMPT) > 600:
    print(f"... [{len(USER_PROMPT)} total chars]")

# Generate
llm = ChatOllama(model="mistral", temperature=0)

response = llm.invoke([
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=USER_PROMPT)
])

print("\n" + "=" * 60)
print("LLM RESPONSE")
print("=" * 60)
print(response.content)

  [Doc 1] NGC4993 — 161 words (full document)
  [Doc 2] M87 — 99 words (full document)
  [Doc 3] Arp273 — 129 words (full document)

Total context: 401 words from 3 documents

SYSTEM PROMPT
You are an astrophysics assistant. Answer using ONLY the provided documents. Cite sources inline with [Doc n]. If the documents do not contain enough information, reply exactly: I don't know.

USER PROMPT (444 words, showing first 600 chars)
Question:
Identify the host galaxy of the gravitational-wave event GW170817 in the provided sample and make          a short summary about properties and identification of GRB

Documents:
[Doc 1] (title=NGC4993, similarity=0.751)
The elliptical galaxy NGC 4993, about 130 million light-years from Earth, viewed with the VIMOS instrument on the European Southern Observatory's Very Large Telescope in Chile.After the almost simultaneous detection of gravitational waves by the LIGO/Virgo collaboration, GW170817, and of a gamma-ray burst by ESA's INTEGRAL and NASA's Fe

### Step 4: Visualize the MOCs cited by the LLM

We parse the `[Doc n]` citations from the LLM response and display **only the MOCs it actually used** in ipyaladin. This closes the loop: query → retrieval → generation → spatial visualization of the answer.

In [16]:
import re
from ipyaladin import Aladin
from IPython.display import display
import ipywidgets as widgets

# Parse which docs the LLM cited
used_nums = set(map(int, re.findall(r'\[Doc\s*(\d+)\]', response.content)))
print(f"LLM cited documents: {sorted(used_nums)}\n")

used_results = [(m, s, f) for i, (m, s, f) in enumerate(results, 1) if i in used_nums]
if not used_results:
    print("LLM did not cite any document — showing all retrieved results.")
    used_results = results

# Show in ipyaladin
aladin = Aladin(target='Sgr A*', fov=180, survey='CDS/P/DSS2/color')
display(aladin)

colors = ['magenta', 'cyan', 'yellow', 'lime', 'orange']
info_rows = []

for i, (moc, sim, fn) in enumerate(used_results):
    title = os.path.splitext(fn)[0]
    color = colors[i % len(colors)]
    aladin.add_moc(moc.moc, color=color, edge=True, adaptativeDisplay=False)
    txt_prev = moc.moc_data.get('text', '')[:200] + '...'
    img = moc.moc_data.get('image', 'N/A')
    info_rows.append(widgets.HTML(
        value=f"<b style='color:{color}'>■</b> <b>{title}</b> | sim={sim:.4f}<br>"
              f"<small>{txt_prev}</small><br>"
              f"<small>Image: <a href='{img}' target='_blank'>{img[:80]}...</a></small><hr>"))
display(widgets.VBox(info_rows))

LLM cited documents: [1]



### Step 5: Vision model analysis

For each cited MOC with an image, gemma3:4b classifies the galaxy.

> **Requires:** `ollama pull gemma3:4b`

In [17]:
import requests as req
import base64
import os

vision_llm = ChatOllama(model='gemma3:4b', temperature=0)

for moc, sim, fn in used_results:
    title = os.path.splitext(fn)[0]
    image_url = moc.moc_data.get('image', '')
    if not image_url:
        continue

    print(f"\n{'='*60}")
    print(f"Galaxy: {title} | sim={sim:.4f}")

    try:
        resp = req.get(image_url, timeout=20)
        resp.raise_for_status()
        encoded = base64.b64encode(resp.content).decode('utf-8')
        ctype = resp.headers.get('Content-Type', 'image/jpeg')
        data_uri = f"data:{ctype};base64,{encoded}"
    except Exception as e:
        print(f"  Failed: {e}")
        continue

    r = vision_llm.invoke([
        SystemMessage(content=(
            "You are an astrophysics assistant. "
            "Classify galaxy morphology conservatively. "
            "If ambiguous, answer uncertain. Be concise."
        )),
        HumanMessage(content=[
            {
                'type': 'text',
                'text': (
                    "Choose exactly one label: "
                    "elliptical | lenticular/S0 | spiral | irregular | uncertain.\n"
                    "Then report exactly in this format:\n"
                    "Label: <label>\n"
                    "Confidence: <0-100>%\n"
                )
            },
            {
                'type': 'image_url',
                'image_url': {'url': data_uri}
            }
        ])
    ])

    print(f"  VISION: {r.content}")
#https://astronomy.swin.edu.au/cosmos/e/early-type+galaxies


Galaxy: NGC4993 | sim=0.7507
  VISION: Elliptical: 95%


> **Note:** The verification of the validity of the multimodels is beyond the scope of this work, which solely illustrates how IVOA standards and tools can be used in the multimodal generative AI systems. In any case, it provides a useful starting point for evaluating their behavior in controlled astronomical environments.

### Step 6: Spatial query

| Object | RA | Dec |
|--------|-----|-----|
| M87 | 187.706 | 12.391 |
| NGC 4993 | 197.450 | -23.381 |
### Try your own prompt

In [18]:
ra_deg, dec_deg = 197.450, -23.381

In [19]:
from astropy.coordinates import Angle

print(f"Searching MOCs at RA={ra_deg}, Dec={dec_deg}\n")
spatial_results = []

for moc, fn in zip(mocs, filenames):
    title = os.path.splitext(fn)[0]
    try: inside = bool(np.asarray(moc.moc.contains_lonlat(
        lon=Angle([ra_deg%360.0], unit='deg'), lat=Angle([dec_deg], unit='deg'))).ravel()[0])
    except: inside = False
    print(f"  {'✅' if inside else '  '} {fn} ({title})")
    if inside: spatial_results.append((moc, 1.0, fn))
print(f"\n{len(spatial_results)} MOC(s)")

if spatial_results:
    a = Aladin(target=f'{ra_deg} {dec_deg}', fov=2, survey='CDS/P/DSS2/color')
    display(a)
    for m, _, f in spatial_results: a.add_moc(m.moc, color='magenta', edge=True)




Searching MOCs at RA=197.45, Dec=-23.381

     Arp273.json (Arp273)
     M101.json (M101)
     M104.json (M104)
     M51.json (M51)
     M59.json (M59)
     M60.json (M60)
     M82.json (M82)
     M87.json (M87)
     NGC4038.json (NGC4038)
     NGC4676.json (NGC4676)
  ✅ NGC4993.json (NGC4993)

1 MOC(s)


## Interactive Chunk Size & Embedding Explorer

Explore how **chunk size**, **embedding model**, and **query formulation** affect retrieval.

In [20]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def run_chunked_search(chunk_size, query, emb_model_name):
    emb_model = OllamaEmbeddings(model=emb_model_name)
    c_mocs, c_fns, c_texts = [], [], []
    for moc, fn, txt in zip(mocs, filenames, texts):
        words = txt.split()
        for i in range(0, len(words), chunk_size):
            c_mocs.append(moc); c_fns.append(fn)
            c_texts.append(' '.join(words[i:i+chunk_size]))
    total = len(c_texts)
    print(f"Model: {emb_model_name}\nChunk size: {chunk_size} words → {total} chunks\n")
    emb_list = []
    for i, t in enumerate(c_texts):
        emb_list.append(emb_model.embed_query(t))
        if (i+1) % 5 == 0 or i == total-1: print(f"\r  Embedding... {i+1}/{total}", end="")
    print(" ✅\n")
    emb = np.array(emb_list, dtype='float32')
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    q = np.array(emb_model.embed_query(query), dtype='float32')
    q = q / np.linalg.norm(q)
    sims = emb @ q
    ranked = sorted(enumerate(sims), key=lambda x: -x[1])
    print(f"Query: '{query}'\n")
    print(f"{'Rank':<6}{'File':<20}{'Sim':<8}{'Words':<8}{'Matched chunk'}")
    print('-' * 100)
    seen, r = set(), 0
    for idx, sim in ranked:
        fn = c_fns[idx]
        if fn in seen: continue
        seen.add(fn)
        if sim < 0.3: continue
        r += 1
        print(f"{r:<6}{fn:<20}{sim:<8.4f}{len(c_texts[idx].split()):<8}{c_texts[idx][:65]}...")
        if r >= 5: break

example_queries = widgets.Dropdown(options=[
    ('— Choose an example —', ''),
    ('── Short ──', ''),
    ('galaxy merger', 'galaxy merger'), ('black hole', 'black hole'), ('kilonova', 'kilonova'),
    ('── Descriptive ──', ''),
    ('elliptical galaxy with supermassive black hole', 'elliptical galaxy with supermassive black hole'),
    ('find the host galaxy of gw170817', 'find the host galaxy of gw170817'),
    ('── Long ──', ''),
    ('elliptical galaxy in Virgo with relativistic jet from supermassive black hole',
     'elliptical galaxy in Virgo with relativistic jet from supermassive black hole'),
    ('── Near-verbatim ──', ''),
    ('NGC 4993 about 130 million light-years from Earth', 'NGC 4993 about 130 million light-years from Earth'),
    ], value='', description='Examples:', style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))
query_input = widgets.Text(value='find the host galaxy of gw170817', description='Query:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))

def on_ex(c):
    if c['new'] and not c['new'].startswith('—') and not c['new'].startswith('──'): query_input.value = c['new']
example_queries.observe(on_ex, names='value')
model_dd = widgets.Dropdown(options=[('nomic-embed-text (768d)', 'nomic-embed-text'),
    ('mxbai-embed-large (1024d)', 'mxbai-embed-large'), ('snowflake-arctic-embed (1024d)', 'snowflake-arctic-embed')],
    value='mxbai-embed-large', description='Model:', style={'description_width': 'initial'}, layout=widgets.Layout(width='600px'))
chunk_sl = widgets.IntSlider(value=200, min=50, max=500, step=25, description='Chunk size:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='500px'))
btn = widgets.Button(description='🔍 Search', button_style='primary', layout=widgets.Layout(width='150px'))
status = widgets.HTML(value='')
output = widgets.Output()

def on_click(b):
    btn.disabled = True; status.value = "<b style='color:orange'>⏳ Computing...</b>"
    with output:
        clear_output()
        try: run_chunked_search(chunk_sl.value, query_input.value, model_dd.value); status.value = "<b style='color:green'>✅ Done</b>"
        except Exception as e: print(f'Error: {e}'); status.value = f"<b style='color:red'>❌ {e}</b>"
    btn.disabled = False
btn.on_click(on_click)
display(widgets.VBox([widgets.HTML('<h3>Interactive Chunk Size & Embedding Explorer</h3>'),
    example_queries, query_input, model_dd, chunk_sl, widgets.HBox([btn, status]), output]))
btn.disabled = True; status.value = "<b style='color:orange'>⏳ Computing...</b>"
with output:
    run_chunked_search(chunk_sl.value, query_input.value, model_dd.value)
    status.value = "<b style='color:green'>✅ Done</b>"
btn.disabled = False

## Agentic Workflow with LLM-Driven Tool Selection

This section implements a real **agentic pipeline** where the LLM decides which tool to use — not a keyword-based router.

The workflow:
1. **Semantic search** — always runs, finds relevant MOCs
2. **LLM decision** — Mistral reads the query and the available tools, then decides which operation to apply
3. **Human in the loop** — you confirm or reject the LLM's decision before execution
4. **Tool execution** — union or intersection of MOCs (from [MOCPy](https://cds-astro.github.io/mocpy/stubs/mocpy.MOC.html))
5. **LLM summary** — generate a scientific description of the result
6. **New Textual MOC** — encapsulate the output as a new Textual MOC

Available MOC operations:
- **Union** — merge sky regions (logical OR): the combined footprint of all selected MOCs
- **Intersection** — overlapping sky region (logical AND): only the area shared by all selected MOCs

If the LLM determines that no tool is appropriate for the query, it says so — no tool is forced.

### Setup: embedding index for the agentic pipeline

In [21]:
from langchain_ollama import OllamaEmbeddings as ollama_embeddings

# Load MOCs with metadata
loaded_mocs = []

for text_moc_gal, text_file, multimedia_url, image in zip(
    text_moc_gals, text_files, multimedia_urls, images):
    file_path = os.path.join(mocs_directory, text_moc_gal)
    tm = TextualMOC(); tm.load_textual_moc(file_path)
    
    loaded_mocs.append({'name': text_moc_gal, 'path': file_path,
        'text': text_file, 'multimedia_url': multimedia_url,
        'image': image, 'moc': tm.moc})

# Setting Embedding model
emb_model = ollama_embeddings(
    model='snowflake-arctic-embed')

docs = [item['text'] for item in loaded_mocs]

print('Computing embeddings...')

doc_emb = np.array(emb_model.embed_documents(docs), dtype='float32')

norms = np.linalg.norm(doc_emb, axis=1, keepdims=True); norms[norms==0]=1e-12

doc_emb = doc_emb / norms

print(f'Embeddings: {doc_emb.shape}')

Textual MOC loaded from moc_gals/Arp273.json.
Textual MOC loaded from moc_gals/M59.json.
Textual MOC loaded from moc_gals/NGC4676.json.
Textual MOC loaded from moc_gals/M101.json.
Textual MOC loaded from moc_gals/M60.json.
Textual MOC loaded from moc_gals/NGC4993.json.
Textual MOC loaded from moc_gals/M104.json.
Textual MOC loaded from moc_gals/M82.json.
Textual MOC loaded from moc_gals/NGC4038.json.
Textual MOC loaded from moc_gals/M51.json.
Textual MOC loaded from moc_gals/M87.json.
Computing embeddings...
Embeddings: (11, 1024)


### Define the tools

Each tool has a name, a description (shown to the LLM), and a function. The LLM reads these descriptions to decide which tool to apply.

In [22]:
def tool_semantic_search(query, top_k=5, threshold=0.25):
    """Search MOCs by semantic similarity to a text query."""
    q = np.array(emb_model.embed_query(query), dtype='float32')
    q = q / max(np.linalg.norm(q), 1e-12)
    sims = doc_emb @ q
    ranked = sorted(enumerate(sims), key=lambda x: -x[1])
    results = []
    for idx, sim in ranked:
        if sim < threshold: continue
        results.append({'idx': idx, 'name': loaded_mocs[idx]['name'],
                        'score': float(sim), 'item': loaded_mocs[idx]})
        if len(results) >= top_k: break
    return results

def tool_moc_union(items, save_path='moc_union.fits'):
    """Compute the union of multiple MOCs (combined sky coverage, logical OR)."""
    if len(items) < 2:
        print('  Need at least 2 MOCs for union.')
        return None
    result = items[0]['moc']
    for item in items[1:]:
        result = result.union(item['moc'])
    result.save(save_path, overwrite=True)
    print(f'  Union saved to: {save_path}')
    return result

def tool_moc_intersection(items, save_path='moc_intersection.fits'):
    """Compute the intersection of multiple MOCs (overlapping sky region, logical AND)."""
    if len(items) < 2:
        print('  Need at least 2 MOCs for intersection.')
        return None
    result = items[0]['moc']
    for item in items[1:]:
        result = result.intersection(item['moc'])
    if result.empty():
        print('  Intersection is empty — these MOCs do not overlap.')
        return None
    result.save(save_path, overwrite=True)
    print(f'  Intersection saved to: {save_path}')
    return result

# Tool registry — the LLM sees these descriptions
TOOLS = {
    'union': {
        'function': tool_moc_union,
        'description': 'Merge multiple MOCs into one covering all their sky regions (logical OR).'
    },
    'intersection': {
        'function': tool_moc_intersection,
        'description': 'Find the overlapping sky region shared by multiple MOCs (logical AND).'
    }
}

TOOL_DESCRIPTIONS = '\n'.join(
    [f'  - {name}: {info["description"]}' for name, info in TOOLS.items()])

print('Available tools:')
print(TOOL_DESCRIPTIONS)

Available tools:
  - union: Merge multiple MOCs into one covering all their sky regions (logical OR).
  - intersection: Find the overlapping sky region shared by multiple MOCs (logical AND).


### The agent: LLM decision + human in the loop

The `agent_decide()` function sends the query and tool descriptions to the LLM. The LLM responds with which tool to use and why. Then `run_agent()` asks you to confirm before executing.

In [23]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

agent_llm = ChatOllama(model='mistral', temperature=0)

def agent_decide(query):
    """Ask the LLM which tool to use."""
    decision_prompt = f"""Given the following user query and the available tools,
decide which tool (if any) should be applied to the retrieved MOCs.

Available tools:
{TOOL_DESCRIPTIONS}
  - none: No spatial operation needed — just retrieve and summarize.

User query: "{query}"

Respond with EXACTLY two lines:
TOOL: <tool_name>
REASON: <one sentence explaining your choice>
"""
    
    response = agent_llm.invoke([
        SystemMessage(content='You decide which spatial operation to apply. Respond only with TOOL and REASON lines.'),
        HumanMessage(content=decision_prompt)
    ])
    text = response.content.strip()
    tool_name, reason = 'none', ''
    for line in text.split('\n'):
        line = line.strip()
        if line.upper().startswith('TOOL:'): tool_name = line.split(':',1)[1].strip().lower()
        elif line.upper().startswith('REASON:'): reason = line.split(':',1)[1].strip()
    return tool_name, reason, text


def run_agent(query, top_k=5, threshold=0.25):
    """Full agentic pipeline with human-in-the-loop."""
    print(f'Query: {query}')
    print('=' * 70)

    # 1. Semantic search (always runs)
    print('\n[1] Semantic search...')
    results = tool_semantic_search(query, top_k=top_k, threshold=threshold)
    if not results:
        print('  No results above threshold.')
        return {'results': [], 'tool': None, 'tool_result': None}
    print(f'  Found {len(results)} relevant MOCs:')
    for i, r in enumerate(results, 1):
        print(f'    {i}. {r["name"]:<20} sim={r["score"]:.4f}')

    # 2. LLM decides which tool to use
    print('\n[2] LLM deciding which tool to use...')
    tool_name, reason, raw = agent_decide(query)
    print(f'  LLM says: {tool_name}')
    print(f'  Reason: {reason}')

    if tool_name == 'none' or tool_name not in TOOLS:
        if tool_name != 'none':
            print(f'  Tool \'{tool_name}\' not available.')
        print('  No tool will be applied.')
        return {'results': results, 'tool': None, 'tool_result': None}

    # 3. Human in the loop
    print(f'\n[3] Human in the loop')
    print(f'  The agent wants to apply: {tool_name}')
    print(f'  Description: {TOOLS[tool_name]["description"]}')
    print(f'  On {len(results)} MOCs: {", ".join(r["name"] for r in results)}')
    confirm = input('\n  Proceed? [y/n] (default=y): ').strip().lower()
    if confirm == 'n':
        print('  Cancelled by user.')
        return {'results': results, 'tool': tool_name, 'tool_result': None, 'cancelled': True}

    # 4. Execute tool
    print(f'\n[4] Executing: {tool_name}...')
    save_path = f'moc_{tool_name}.fits'
    tool_result = TOOLS[tool_name]['function']([r['item'] for r in results], save_path=save_path)

    return {'results': results, 'tool': tool_name, 'tool_result': tool_result, 'save_path': save_path}

### Run the agent

Try different queries:
- `"Find spiral galaxies and merge their sky coverage"` → LLM should pick **union**
- `"Find the overlapping region between elliptical galaxies"` → LLM should pick **intersection**
- `"Tell me about galaxy mergers"` → LLM should pick **none** (just retrieve, no spatial operation)

In [31]:
out = run_agent('Search the database for the most well-known spiral galaxies and merge their sky coverage ', top_k=3, threshold=0.6)

Query: Search the database for the most well-known spiral galaxies and merge their sky coverage 

[1] Semantic search...
  Found 3 relevant MOCs:
    1. Arp273.json          sim=0.7433
    2. M51.json             sim=0.6895
    3. NGC4676.json         sim=0.6676

[2] LLM deciding which tool to use...
  LLM says: union
  Reason: To combine the sky regions of the most well-known spiral galaxies for a comprehensive view.

[3] Human in the loop
  The agent wants to apply: union
  Description: Merge multiple MOCs into one covering all their sky regions (logical OR).
  On 3 MOCs: Arp273.json, M51.json, NGC4676.json



  Proceed? [y/n] (default=y):  y



[4] Executing: union...
  Union saved to: moc_union.fits


In [25]:
# Try another query — the LLM should pick intersection
#out = run_agent('Find the overlapping region between elliptical galaxies', top_k=5, threshold=0.25)

### Visualize the result in ipyaladin

In [32]:
if out.get('tool_result') is not None:
    aladin = Aladin(target='10 00 +02 12', fov=180, survey='CDS/P/DSS2/color')
    display(aladin)
    colors = ['magenta', 'cyan', 'yellow', 'lime', 'orange']
    for i, r in enumerate(out['results']):
        aladin.add_moc(r['item']['moc'], color=colors[i % len(colors)],
                       edge=True, adaptativeDisplay=False)
    tool_color = 'red' if out['tool'] == 'union' else 'blue'
    aladin.add_moc(out['tool_result'], color=tool_color,
                   edge=True, adaptativeDisplay=False)
    print(f"{tool_color.capitalize()} layer = MOC {out['tool']}")
else:
    print('No tool was applied — nothing to visualize.')

Red layer = MOC union


### Generate LLM summary and create a new Textual MOC

In [33]:
if out.get('tool_result') is not None:
    selected = [r['item'] for r in out['results']]
    combined = '\n\n'.join([f"Galaxy {i+1}: {it['name']}\n{it['text']}"
                             for i, it in enumerate(selected)])
    summary_prompt = f"""
Summarize these astronomical galaxy descriptions in 6-8 sentences.
Focus on: galaxy types, characteristics, what the {out['tool']} sky coverage represents.

Texts:
{combined}
"""
    print('=== PROMPT TO LLM ===')
    print(summary_prompt[:500])
    if len(summary_prompt) > 500: print(f'... [{len(summary_prompt)} chars]')

    summary = agent_llm.invoke(summary_prompt)
    print('\n=== LLM SUMMARY ===')
    print(summary.content)

    # Create new Textual MOC
    textual_result = TextualMOC(out['tool_result'])
    textual_result.add_text_media_image(
        summary.content,
        selected[0].get('multimedia_url', ''),
        selected[0].get('image', ''))
    textual_result.update_metadata(author='Agentic SemanticMOC workflow')
    output_file = f"moc_{out['tool']}_textual.json"
    textual_result.save(output_file)
    print(f'\nNew Textual MOC saved to: {output_file}')

    textual_result.render_ipyaladin(color='red', opacity=0.35,
                                     survey='P/Mellinger/color', fov=180)
else:
    print('No tool result — skipping summary.')

=== PROMPT TO LLM ===

Summarize these astronomical galaxy descriptions in 6-8 sentences.
Focus on: galaxy types, characteristics, what the union sky coverage represents.

Texts:
Galaxy 1: Arp273.json
The galaxies' twisted and distorted appearance is due to mutual gravitational tides as the pair engage in close encounters. Cataloged as Arp 273 (also as UGC 1810), these galaxies do look peculiar, but interacting galaxies are now understood to be common in the universe. Closer to home, the large spiral Andromeda Galaxy
... [2459 chars]

=== LLM SUMMARY ===
1. Arp273, also known as UGC 1810, is a pair of interacting galaxies with a peculiar appearance due to gravitational tides during close encounters. They serve as an analog for the future encounter between Andromeda and the Milky Way, eventually merging into a single galaxy after repeated interactions on a cosmic timescale. From our perspective, their bright cores are separated by approximately 100,000 light-years.

2. M51, or the Whirl

In [29]:
# Only text, in particular for no space operation
if out.get('tool_result') is not None:
    print(textual_result.moc_data.get('text', ''))

1. Arp273 (UGC 1810) is a pair of interacting galaxies with a peculiar appearance due to close encounters and mutual gravitational tides. They serve as an analog for the future encounter between Andromeda and Milky Way, eventually merging into a single galaxy after repeated encounters on a cosmic timescale.

2. M87 is an elliptical galaxy located in the Virgo constellation, approximately 54 million light-years away from Earth. It hosts several trillion stars, a supermassive black hole, and around 15,000 globular star clusters, making it significantly larger than our Milky Way.

3. NGC4676, or 'The Mice,' is a colliding pair of spiral galaxies that will eventually merge into a single giant galaxy. The interaction triggers the formation of young, hot blue stars and long tails of stars and gas between the two galaxies.

4. M60 is an elliptical galaxy within the Virgo cluster, which contains over 1,300 galaxies. It has a diameter of 120,000 light-years, a huge black hole at its center, and

## Concluding Remarks

This tutorial has explored how Multi-Order Coverage maps can evolve from compact spatial descriptors into semantically enriched and operational scientific objects. By progressively combining textual enrichment, vector representations, semantic retrieval, tool-based MOC operations, and LLM-driven summarization, we have shown that the value of Semantic MOCs lies not only in improved discoverability, but in their ability to support complete knowledge workflows grounded in astronomical data.

Semantic MOCs provide a common representation through which spatial information, semantic similarity, and downstream computational actions can be linked within a single pipeline. This makes it possible to move beyond static annotation and toward interactive, interpretable, and extensible workflows in which natural-language queries can trigger both retrieval and domain-specific operations.

More broadly, this perspective suggests that Semantic MOCs may serve as a practical interface between established astronomical standards and emerging GenAI systems. Rather than treating language models as external components loosely attached to scientific archives, this framework integrates them into a structured representation that preserves domain semantics and supports grounded reasoning over sky regions and associated metadata.

We therefore see Semantic MOCs as a promising step toward a new class of astronomy-aware AI workflows, in which spatial coverage, textual knowledge, and computational tools can be jointly orchestrated. Future work may extend this direction through larger-scale indexing, richer multimodal enrichment, standardized interoperability, and more explicit agentic tool use over scientific data products.

In [30]:
tool_name, reason, raw = agent_decide(query)
print(f"  Raw LLM response:\n  {raw}")  # ← aggiungi questo
print(f"  LLM says: {tool_name}")
print(f"  Reason: {reason}")

  Raw LLM response:
  TOOL: intersection
   REASON: To find the overlapping region between the MOCs of GW170817 and any potential host galaxy candidates.
  LLM says: intersection
  Reason: To find the overlapping region between the MOCs of GW170817 and any potential host galaxy candidates.


Note: This manual tool-routing pattern is equivalent to the function calling mechanism available in commercial APIs (OpenAI, Anthropic). We implement it explicitly here for transparency. The LLM receives the tool descriptions as text and decides which one to apply — the same principle used by frameworks like LangChain and LangGraph.

---

### How tool selection works in LLM agents

In modern LLM-based systems, **tool use** (also called *function calling*) is a standard pattern: the model receives a description of available tools and autonomously decides which one to invoke based on the user's query.

This is the same mechanism used by commercial APIs (OpenAI function calling, Anthropic tool use) and frameworks like LangChain and LangGraph. The difference is that those systems use structured JSON schemas and automatic parsing, while here we implement it explicitly with a text prompt — for full transparency.

**The process in detail:**
```
User query
    │
    ▼
┌──────────────────────────────────┐
│  LLM receives:                   │
│  • the user query                │
│  • a list of available tools     │
│    with their descriptions       │
│  • instructions to respond with  │
│    TOOL: <name> and REASON: <why>│
└──────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────┐
│  LLM responds:                   │
│  TOOL: union                     │
│  REASON: The user wants to       │
│  combine sky coverages.          │
└──────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────┐
│  Human in the loop:              │
│  "The agent wants to apply       │
│   union on 5 MOCs. Proceed?"     │
│  [y/n]                           │
└──────────────────────────────────┘
    │
    ▼
  Tool executes (or not)
```

**Why this works:** The LLM does not match keywords — it reads the *meaning* of the query against the *meaning* of each tool description. For example, all of the following queries would trigger the `union` tool:

- *"merge their sky coverage"*
- *"combine the footprints"*
- *"join these regions into one"*
- *"create a single MOC from all of them"*

The LLM understands that these all express the same intent, even though none of them contains the word "union".

**Why human in the loop:** In a scientific context, autonomous tool execution carries risk — merging or intersecting MOCs changes the data product. The confirmation step ensures the researcher retains control over what operations are applied. This is a deliberate design choice: the LLM *proposes*, the human *approves*.

---

---

### How the RAG pipeline works in this tutorial

**Retrieval-Augmented Generation (RAG)** is a technique that grounds LLM responses in real data. Instead of asking the model a question directly — where it might hallucinate — we first *retrieve* relevant documents and then pass them as context to the LLM, which *generates* an answer based only on what it received.

Our implementation has a specific design choice: **chunked retrieval with full-document generation**. The text of each MOC is split into smaller segments (chunks) for the search step, but the complete document is passed to the LLM for the generation step.

**The process in detail:**
```
User query: "find the host galaxy of gw170817"
    │
    ▼
┌──────────────────────────────────────┐
│  1. CHUNKING                         │
│  Each MOC text is split into chunks  │
│  of ~100 words. A 300-word document  │
│  becomes 3 chunks, each capturing    │
│  a specific aspect of the text.      │
│                                      │
│  NGC4993 chunk 1: "The elliptical    │
│  galaxy NGC 4993, about 130..."      │
│  NGC4993 chunk 2: "...gravitational  │
│  waves, neutron stars, kilonova..."  │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│  2. EMBEDDING + SIMILARITY SEARCH    │
│  Both the query and all chunks are   │
│  converted to vectors with the same  │
│  model (mxbai-embed-large).          │
│  Cosine similarity finds the chunks  │
│  closest in meaning to the query.    │
│                                      │
│  "gw170817" ←→ "gravitational waves, │
│  neutron stars, kilonova" = HIGH sim │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│  3. CHUNK → PARENT DOCUMENT          │
│  The matched chunk identifies which  │
│  MOC is relevant. But we pass the    │
│  FULL document to the LLM, not the   │
│  chunk — so the model has complete   │
│  context to answer well.             │
│                                      │
│  chunk found: NGC4993 chunk 2        │
│  passed to LLM: entire NGC4993 text  │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│  4. LLM GENERATION                   │
│  The LLM receives:                   │
│  • a system prompt (role, rules)     │
│  • the full documents as [Doc 1],    │
│    [Doc 2], etc.                     │
│  • the user question                 │
│  • instruction to cite sources       │
│                                      │
│  The LLM answers using ONLY the      │
│  provided documents, citing them     │
│  with [Doc n] tags.                  │
└──────────────────────────────────────┘
    │
    ▼
┌──────────────────────────────────────┐
│  5. VISUALIZATION                    │
│  We parse the [Doc n] citations      │
│  from the LLM response and display   │
│  only the cited MOCs in ipyaladin —  │
│  showing both spatial coverage and   │
│  text in a single view.              │
└──────────────────────────────────────┘
```

**Why chunk for search but pass full documents?** A chunk like *"gravitational waves, neutron stars, kilonova"* is semantically close to the query *"gw170817"* — but it is too short for the LLM to generate a complete answer. The full document contains coordinates, distances, discovery context, and other details the LLM needs. The chunk is a *search key*; the document is the *knowledge*.

**Why not just embed full documents?** Because long texts dilute specific information. The term *"kilonova"* in a 300-word document contributes little to the overall embedding vector — it gets averaged out with all the other words. In a 100-word chunk, that same term dominates the vector and drives a strong similarity match.

**No vector database in this tutorial.** The embeddings are computed in memory with numpy. For 11 MOCs this is instantaneous. A vector database (FAISS, Weaviate, Chroma) becomes necessary when the collection grows to thousands or millions of documents, where you need persistent storage, metadata filtering, and efficient approximate nearest-neighbor search.

---

---

### From spatial coverage to semantic objects: the MOC hierarchy

A **Multi-Order Coverage map (MOC)** is an IVOA standard that describes a region of the sky as a set of HEALPix cells at multiple resolutions. It is a compact, hierarchical representation — a MOC can encode the footprint of an entire survey in a few kilobytes.

In this tutorial series, we progressively extend the MOC from a purely spatial descriptor into a semantically enriched scientific object.

**The hierarchy:**
```
┌─────────────────────────────────────────────────┐
│  MOC (IVOA standard)                            │
│  A set of HEALPix cells describing a sky region │
│  • Spatial coverage only                        │
│  • No text, no metadata, no meaning             │
│  • Example: "this MOC covers 12 sq deg around   │
│    RA=187.7, Dec=12.4"                          │
└─────────────────────────────────────────────────┘
        │
        │  + text, media, image, annotations, metadata
        ▼
┌─────────────────────────────────────────────────┐
│  Textual MOC                                    │
│  A MOC enriched with human-readable content     │
│  • Text description of the sky region           │
│  • Image URL (e.g. from hips2fits)              │
│  • Multimedia link (e.g. NASA page)             │
│  • Cell annotations (labels on HEALPix cells)   │
│  • Author, date metadata                        │
│  • Example: "M87 is a giant elliptical galaxy   │
│    hosting a supermassive black hole..."         │
│  • Stored as a single JSON file                 │
└─────────────────────────────────────────────────┘
        │
        │  + embedding vector
        ▼
┌─────────────────────────────────────────────────┐
│  Semantic MOC                                   │
│  A Textual MOC enriched with a vector embedding │
│  • 768 or 1024-dimensional vector encoding the  │
│    meaning of the text                          │
│  • Enables semantic search: find MOCs by        │
│    meaning, not just by position or keywords    │
│  • Self-contained: the embedding travels with   │
│    the MOC in the same JSON file                │
│  • Example: the vector for M87 is close to the  │
│    vector for "supermassive black hole jet"      │
└─────────────────────────────────────────────────┘
```

**What each layer enables:**

| Layer | You can ask | Method |
|-------|------------|--------|
| MOC | "Does this position fall inside the region?" | `contains_lonlat(ra, dec)` |
| Textual MOC | "What is this region about?" | `show_text_value()`, `render_ipyaladin()` |
| Semantic MOC | "Find regions similar in meaning to my query" | Cosine similarity on embeddings |

**Why this matters:** Traditional astronomical archives let you search by position (cone search) or by keywords (text match). A Semantic MOC adds a third dimension: **search by meaning**. The query *"neutron star merger electromagnetic counterpart"* finds NGC 4993 even if those exact words do not appear in the description — because the embedding captures the semantic proximity between the query and the text.

**The key innovation:** The embedding is stored *inside* the MOC JSON, not in an external database. This makes the Semantic MOC **self-contained and portable** — it carries its own spatial coverage, textual description, and semantic vector in a single file. You can share it, archive it, or index it without any external dependency.
```
Semantic MOC JSON structure:
{
  "moc": "3/45 67 89 4/120...",       ← HEALPix cells (spatial)
  "text": "M87 is a giant...",         ← description (textual)
  "image": "https://alasky...",        ← image URL
  "multimedia_url": "https://...",     ← reference link
  "author": "G. Greco",               ← metadata
  "date": "2026-03-18",
  "embedding": [0.12, -0.45, ...],    ← 768-dim vector (semantic)
  "embedding_model": "nomic-embed-text"
}
```

This single JSON file is simultaneously a spatial descriptor, a textual document, and a vector in embedding space. It can be queried spatially (`contains_lonlat`), read by a human (`show_text_value`), and retrieved by an LLM (`cosine similarity`).

---